# Microscope Examples

This page provides concrete examples of how to use `ndx-ophys-devices` to describe the hardware metadata of standard microscope systems used in *in vivo* optical physiology experiments. Each example reflects a realistic configuration based on commercially available components.

Four common setups are covered:

1. [Two-Photon Laser Scanning Microscope](#1-two-photon-laser-scanning-microscope-2plsm)
2. [One-Photon Widefield Calcium Imaging](#2-one-photon-widefield-calcium-imaging)
3. [Fiber Photometry System](#3-fiber-photometry-system)
4. [Head-Mounted Miniscope](#4-head-mounted-miniscope-ucla-miniscope-v4)

In [ ]:
import datetime
from pynwb import NWBFile, NWBHDF5IO

from ndx_ophys_devices import (
    StereotacticPosition,
    ViralVector,
    ViralVectorInjection,
    Indicator,
    ExcitationSourceModel,
    PhotodetectorModel,
    DichroicMirrorModel,
    BandOpticalFilterModel,
    ObjectiveLensModel,
    OpticalFiberModel,
    ExcitationSource,
    PulsedExcitationSource,
    Photodetector,
    DichroicMirror,
    BandOpticalFilter,
    ObjectiveLens,
    OpticalFiber,
)


## 1. Two-Photon Laser Scanning Microscope (2PLSM)

Two-photon laser scanning microscopy is the most widely used technique for *in vivo* calcium imaging of neuronal populations in head-fixed animals. It offers sub-cellular resolution at depths up to ~500 µm in scattering tissue.

**Typical configuration:**
- Pulsed near-infrared laser (Ti:Sapphire or fiber laser) tuned to ~920 nm for GCaMP excitation
- Water-immersion objective with high NA for light collection
- GaAsP photomultiplier tubes (PMTs) for fluorescence detection
- Bandpass emission filters to separate green (GCaMP) and red (e.g., tdTomato) channels

This example models a system similar to a Bruker Ultima Investigator or Scientifica HyperScope.

In [2]:
nwbfile = NWBFile(
    session_description="Example Two-photon laser scanning microscopy ",
    identifier="example_001",
    session_start_time=datetime.datetime.now(datetime.timezone.utc),
)

In [3]:
# --- Viral vector & indicator ---

viral_vector_2p = ViralVector(
    name="viral_vector_2p",
    description="AAV for sparse GCaMP8s expression via Cre-dependent strategy",
    construct_name="AAV1-Syn-Flex-jGCaMP8s-WPRE",
    manufacturer="Addgene",
    titer_in_vg_per_ml=2.5e13,
)

injection_coords_2p = StereotacticPosition(
    name="viral_injection_coordinates",
    anatomical_target="Primary visual cortex (V1)",
    origin="bregma",
    orientation="RAS",
    x_in_mm=-2.5,   # mediolateral: 2.5 mm lateral
    y_in_mm=3.5,    # anteroposterior: 3.5 mm posterior
    z_in_mm=-0.5,   # dorsoventral: 0.5 mm below surface
)

injection_2p = ViralVectorInjection(
    name="viral_vector_injection",
    description="Cortical injection under isoflurane anesthesia, 3 weeks before imaging",
    volume_in_uL=0.3,
    injection_date="2024-11-01T10:00:00+00:00",
    viral_injection_coordinates=injection_coords_2p,
    viral_vector=viral_vector_2p,
)

indicator_2p = Indicator(
    name="indicator",
    description="Genetically encoded calcium indicator for neuronal activity",
    label="jGCaMP8s",
    manufacturer="Addgene",
    viral_vector_injection=injection_2p,
)

# --- Excitation source: Coherent Chameleon Ultra II (Ti:Sapphire laser) ---
# Tunable 680–1080 nm, ~140 fs pulse width, 80 MHz repetition rate

laser_model_2p = ExcitationSourceModel(
    name="chameleon_ultra_ii_model",
    manufacturer="Coherent",
    model_number="Chameleon Ultra II",
    description=(
        "Mode-locked Ti:Sapphire laser, tunable 680-1080 nm, "
        "~140 fs pulse width, 80 MHz repetition rate"
    ),
    source_type="laser",
    excitation_mode="two-photon",
    wavelength_range_in_nm=[680.0, 1080.0],
)
nwbfile.add_device_model(laser_model_2p)

laser_2p = PulsedExcitationSource(
    name="chameleon_ultra_ii",
    description="Ti:Sapphire laser tuned to 920 nm for GCaMP8s two-photon excitation",
    serial_number="CHU-2024-00123",
    model=laser_model_2p,
    peak_power_in_W=50e3,         # ~50 kW peak power at 100 mW average, 140 fs, 80 MHz
    peak_pulse_energy_in_J=1.25e-9,  # ~1.25 nJ per pulse at 100 mW average, 80 MHz
    intensity_in_W_per_m2=1.5e12,    # typical at focal point
    exposure_time_in_s=1.4e-13,      # ~140 fs pulse width
    pulse_rate_in_Hz=80e6,           # 80 MHz repetition rate
)
nwbfile.add_device(laser_2p)

# --- Objective lens: Olympus 25x XLPLN25XWMP2 ---
# 25x water-immersion, NA 1.05, WD 2 mm

objective_model_2p = ObjectiveLensModel(
    name="xlpln25xwmp2_model",
    manufacturer="Olympus",
    model_number="XLPLN25XWMP2",
    description="25x water-immersion objective, NA 1.05, working distance 2 mm",
    numerical_aperture=1.05,
    magnification=25.0,
)
nwbfile.add_device_model(objective_model_2p)

lens_position_2p = StereotacticPosition(
    name="lens_positioning",
    anatomical_target="Primary visual cortex (V1), layer 2/3",
    origin="bregma",
    orientation="RAS",
    x_in_mm=-2.5,
    y_in_mm=3.5,
    z_in_mm=0.0,   # at brain surface
)

objective_2p = ObjectiveLens(
    name="objective_lens",
    description="Water-immersion objective for two-photon in vivo imaging",
    serial_number="OBJ-2024-00456",
    model=objective_model_2p,
    lens_positioning=lens_position_2p,
)
nwbfile.add_device(objective_2p)

# --- Dichroic mirror: Semrock FF580-FDi01 ---
# 580 nm longpass dichroic, reflects <580 nm, transmits >580 nm

dichroic_model_2p = DichroicMirrorModel(
    name="ff580_fdi01_model",
    manufacturer="Semrock",
    model_number="FF580-FDi01",
    description="580 nm longpass dichroic mirror for green/red channel separation",
    cut_on_wavelength_in_nm=580.0,
    reflection_band_in_nm=[400.0, 562.0],
    transmission_band_in_nm=[598.0, 800.0],
    angle_of_incidence_in_degrees=45.0,
)
nwbfile.add_device_model(dichroic_model_2p)

dichroic_2p = DichroicMirror(
    name="dichroic_mirror",
    description="Emission beam splitter separating green and red fluorescence channels",
    serial_number="DM-2024-00789",
    model=dichroic_model_2p,
)
nwbfile.add_device(dichroic_2p)

# --- Green channel emission filter: Semrock FF03-525/50-25 ---
# 525/50 bandpass (500–550 nm), optimized for GCaMP emission

green_filter_model_2p = BandOpticalFilterModel(
    name="ff03_525_50_model",
    manufacturer="Semrock",
    model_number="FF03-525/50-25",
    description="525/50 nm bandpass emission filter for GCaMP fluorescence detection",
    filter_type="Bandpass",
    center_wavelength_in_nm=525.0,
    bandwidth_in_nm=50.0,
)
nwbfile.add_device_model(green_filter_model_2p)

green_filter_2p = BandOpticalFilter(
    name="green_emission_filter",
    description="Bandpass filter in green detection channel (GCaMP8s emission)",
    serial_number="GF-2024-00101",
    model=green_filter_model_2p,
)
nwbfile.add_device(green_filter_2p)

# --- Red channel emission filter: Semrock FF01-605/70-25 ---
# 605/70 bandpass (570–640 nm), for red structural markers (e.g., tdTomato)

red_filter_model_2p = BandOpticalFilterModel(
    name="ff01_605_70_model",
    manufacturer="Semrock",
    model_number="FF01-605/70-25",
    description="605/70 nm bandpass emission filter for red fluorophore detection",
    filter_type="Bandpass",
    center_wavelength_in_nm=605.0,
    bandwidth_in_nm=70.0,
)
nwbfile.add_device_model(red_filter_model_2p)

red_filter_2p = BandOpticalFilter(
    name="red_emission_filter",
    description="Bandpass filter in red detection channel (tdTomato structural marker)",
    serial_number="RF-2024-00102",
    model=red_filter_model_2p,
)
nwbfile.add_device(red_filter_2p)

# --- Photodetector: Hamamatsu H10770PA-40 (GaAsP PMT) ---
# High-sensitivity GaAsP photomultiplier tube, 300–720 nm, ~40% QE at 500 nm

pmt_model_2p = PhotodetectorModel(
    name="h10770pa_40_model",
    manufacturer="Hamamatsu",
    model_number="H10770PA-40",
    description=(
        "GaAsP photomultiplier tube, spectral response 300-720 nm, "
        "~40% quantum efficiency at 500 nm"
    ),
    detector_type="PMT",
    wavelength_range_in_nm=[300.0, 720.0],
    gain=1e6,
    gain_unit="V/W",
)
nwbfile.add_device_model(pmt_model_2p)

pmt_green_2p = Photodetector(
    name="pmt_green_channel",
    description="GaAsP PMT for green fluorescence channel (GCaMP8s)",
    serial_number="PMT-2024-00201",
    model=pmt_model_2p,
)
nwbfile.add_device(pmt_green_2p)

pmt_red_2p = Photodetector(
    name="pmt_red_channel",
    description="GaAsP PMT for red fluorescence channel (structural marker)",
    serial_number="PMT-2024-00202",
    model=pmt_model_2p,
)
nwbfile.add_device(pmt_red_2p)


pmt_red_channel abc.Photodetector at 0x1859853977872
Fields:
  description: GaAsP PMT for red fluorescence channel (structural marker)
  model: h10770pa_40_model abc.PhotodetectorModel at 0x1859854963040
Fields:
  description: GaAsP photomultiplier tube, spectral response 300-720 nm, ~40% quantum efficiency at 500 nm
  detector_type: PMT
  gain: 1000000.0
  gain_unit: V/W
  manufacturer: Hamamatsu
  model_number: H10770PA-40
  wavelength_range_in_nm: [300. 720.]

  serial_number: PMT-2024-00202

In [4]:
with NWBHDF5IO("example_001.nwb", "w") as io:
    io.write(nwbfile)

In [ ]:
with NWBHDF5IO("example_001.nwb", "r") as io:
    read_nwbfile = io.read()
    for name, device in read_nwbfile.devices.items():
        print(name, type(device).__name__)

chameleon_ultra_ii PulsedExcitationSource
dichroic_mirror DichroicMirror
green_emission_filter BandOpticalFilter
objective_lens ObjectiveLens
pmt_green_channel Photodetector
pmt_red_channel Photodetector
red_emission_filter BandOpticalFilter


## 2. One-Photon Widefield Calcium Imaging

Widefield one-photon imaging captures mesoscale fluorescence signals across large cortical areas simultaneously. It is commonly used to map functional connectivity or study cortex-wide dynamics, often using transgenic animals with pan-neuronal GCaMP expression.

**Typical configuration:**
- LED excitation light source with excitation bandpass filter
- Low-magnification, high-NA objective
- Scientific CMOS (sCMOS) camera for large field-of-view detection
- Bandpass emission filter for GCaMP

In [6]:
nwbfile = NWBFile(
    session_description="Example Widefield one-photon imaging",
    identifier="example_002",
    session_start_time=datetime.datetime.now(datetime.timezone.utc),
)

In [7]:
# --- Excitation source: Lumencor SPECTRA X Light Engine ---
# Multi-channel LED light engine; 470 nm channel used for GCaMP excitation

led_model_wf = ExcitationSourceModel(
    name="spectra_x_model",
    manufacturer="Lumencor",
    model_number="SPECTRA X",
    description=(
        "Multi-LED light engine with individually switchable spectral bands; "
        "470 nm channel used for GCaMP excitation"
    ),
    source_type="LED",
    excitation_mode="one-photon",
    wavelength_range_in_nm=[380.0, 680.0],
)
nwbfile.add_device_model(led_model_wf)

led_wf = ExcitationSource(
    name="spectra_x",
    description="LED light engine providing 470 nm excitation for GCaMP widefield imaging",
    serial_number="SPX-2024-00301",
    model=led_model_wf,
    power_in_W=0.5e-3,           # ~0.5 mW at sample
    intensity_in_W_per_m2=5.0,   # typical for widefield illumination
    exposure_time_in_s=50e-3,    # 50 ms exposure per frame
)
nwbfile.add_device(led_wf)

# --- Excitation filter: Chroma ET470/40x ---
# 470/40 bandpass (450–490 nm) for LED excitation cleanup

excitation_filter_model_wf = BandOpticalFilterModel(
    name="et470_40x_model",
    manufacturer="Chroma",
    model_number="ET470/40x",
    description="470/40 nm bandpass excitation filter for GCaMP one-photon excitation",
    filter_type="Bandpass",
    center_wavelength_in_nm=470.0,
    bandwidth_in_nm=40.0,
)
nwbfile.add_device_model(excitation_filter_model_wf)

excitation_filter_wf = BandOpticalFilter(
    name="excitation_filter",
    description="Bandpass filter for LED excitation cleanup (GCaMP channel)",
    serial_number="EF-2024-00401",
    model=excitation_filter_model_wf,
)
nwbfile.add_device(excitation_filter_wf)

# --- Dichroic mirror: Semrock FF495-Di03 ---
# 495 nm longpass dichroic

dichroic_model_wf = DichroicMirrorModel(
    name="ff495_di03_model",
    manufacturer="Semrock",
    model_number="FF495-Di03",
    description="495 nm longpass dichroic mirror separating GCaMP excitation and emission",
    cut_on_wavelength_in_nm=495.0,
    reflection_band_in_nm=[350.0, 484.0],
    transmission_band_in_nm=[506.0, 800.0],
    angle_of_incidence_in_degrees=45.0,
)
nwbfile.add_device_model(dichroic_model_wf)

dichroic_wf = DichroicMirror(
    name="dichroic_mirror",
    description="495 nm dichroic mirror in GCaMP fluorescence cube",
    serial_number="DM-2024-00501",
    model=dichroic_model_wf,
)
nwbfile.add_device(dichroic_wf)

# --- Emission filter: Semrock FF03-535/30-25 ---
# 535/30 bandpass (520–550 nm) for GCaMP emission

emission_filter_model_wf = BandOpticalFilterModel(
    name="ff03_535_30_model",
    manufacturer="Semrock",
    model_number="FF03-535/30-25",
    description="535/30 nm bandpass emission filter optimized for GCaMP fluorescence",
    filter_type="Bandpass",
    center_wavelength_in_nm=535.0,
    bandwidth_in_nm=30.0,
)
nwbfile.add_device_model(emission_filter_model_wf)

emission_filter_wf = BandOpticalFilter(
    name="emission_filter",
    description="Bandpass emission filter for GCaMP widefield signal detection",
    serial_number="EMF-2024-00601",
    model=emission_filter_model_wf,
)
nwbfile.add_device(emission_filter_wf)

# --- Objective: Nikon CFI Plan Apo Lambda 10x ---
# 10x air objective, NA 0.45, optimized for large field-of-view cortical imaging

objective_model_wf = ObjectiveLensModel(
    name="cfi_plan_apo_10x_model",
    manufacturer="Nikon",
    model_number="CFI Plan Apo Lambda 10x",
    description="10x air objective, NA 0.45, for large field-of-view widefield imaging",
    numerical_aperture=0.45,
    magnification=10.0,
)
nwbfile.add_device_model(objective_model_wf)

cortex_position_wf = StereotacticPosition(
    name="lens_positioning",
    anatomical_target="Dorsal cortex",
    origin="bregma",
    orientation="RAS",
    x_in_mm=0.0,
    y_in_mm=0.0,
    z_in_mm=0.0,
)

objective_wf = ObjectiveLens(
    name="objective_lens",
    description="Low-magnification objective for widefield cortical imaging",
    serial_number="OBJ-2024-00701",
    model=objective_model_wf,
    lens_positioning=cortex_position_wf,
)
nwbfile.add_device(objective_wf)

# --- Photodetector: Hamamatsu ORCA-Flash4.0 V3 (sCMOS camera) ---
# Scientific CMOS, 2048x2048, 16-bit, up to 100 fps, 82% peak QE

scmos_model_wf = PhotodetectorModel(
    name="orca_flash4_v3_model",
    manufacturer="Hamamatsu",
    model_number="C13440-20CU",
    description=(
        "ORCA-Flash4.0 V3 sCMOS camera, 2048x2048 pixels, 6.5 µm pixel size, "
        "82% peak quantum efficiency, up to 100 fps"
    ),
    detector_type="sCMOS",
    wavelength_range_in_nm=[350.0, 1100.0],
)
nwbfile.add_device_model(scmos_model_wf)

scmos_wf = Photodetector(
    name="orca_flash4",
    description="sCMOS camera for widefield calcium imaging",
    serial_number="CAM-2024-00801",
    model=scmos_model_wf,
)
nwbfile.add_device(scmos_wf)


orca_flash4 abc.Photodetector at 0x1859853328176
Fields:
  description: sCMOS camera for widefield calcium imaging
  model: orca_flash4_v3_model abc.PhotodetectorModel at 0x1859853986192
Fields:
  description: ORCA-Flash4.0 V3 sCMOS camera, 2048x2048 pixels, 6.5 µm pixel size, 82% peak quantum efficiency, up to 100 fps
  detector_type: sCMOS
  manufacturer: Hamamatsu
  model_number: C13440-20CU
  wavelength_range_in_nm: [ 350. 1100.]

  serial_number: CAM-2024-00801

In [8]:
with NWBHDF5IO("example_002.nwb", "w") as io:
    io.write(nwbfile)
    


In [9]:
with NWBHDF5IO("example_002.nwb", "r") as io:
    read_nwbfile = io.read()
    for name, device in read_nwbfile.devices.items():
        print(name, type(device).__name__)

dichroic_mirror DichroicMirror
emission_filter BandOpticalFilter
excitation_filter BandOpticalFilter
objective_lens ObjectiveLens
orca_flash4 Photodetector
spectra_x ExcitationSource


## 3. Fiber Photometry System

Fiber photometry measures bulk fluorescence from genetically encoded indicators in deep brain structures in freely moving animals. A single multimode optical fiber both delivers excitation light and collects emitted fluorescence.

**Typical configuration (Doric Lenses-style system):**
- Two excitation wavelengths: 473 nm (GCaMP signal) and 405 nm (isosbestic control for motion artifacts)
- Multimode optical fiber chronically implanted above the brain region of interest
- Bandpass emission filter to select GCaMP emission
- Silicon photoreceiver for detection


In [10]:
nwbfile = NWBFile(
    session_description="Example Fiber photometry",
    identifier="example_003",
    session_start_time=datetime.datetime.now(datetime.timezone.utc),
)

In [11]:
# --- Viral vector & indicator ---

viral_vector_fp = ViralVector(
    name="viral_vector_fp",
    description="AAV for GCaMP7f expression in dopaminergic neurons",
    construct_name="AAV5-CAG-dLight1.3b",
    manufacturer="Addgene",
    titer_in_vg_per_ml=5.0e12,
)

injection_coords_fp = StereotacticPosition(
    name="viral_injection_coordinates",
    anatomical_target="Nucleus accumbens (NAc), core",
    origin="bregma",
    orientation="RAS",
    x_in_mm=1.3,    # lateral
    y_in_mm=1.3,    # anterior
    z_in_mm=-4.5,   # ventral
)

injection_fp = ViralVectorInjection(
    name="viral_vector_injection",
    description="Stereotaxic injection targeting NAc core, 3 weeks before recording",
    volume_in_uL=0.5,
    injection_date="2024-10-15T09:00:00+00:00",
    viral_injection_coordinates=injection_coords_fp,
    viral_vector=viral_vector_fp,
)

indicator_fp = Indicator(
    name="indicator",
    description="Dopamine indicator for fiber photometry",
    label="dLight1.3b",
    manufacturer="Addgene",
    viral_vector_injection=injection_fp,
)

# --- Excitation source 1: 473 nm laser (GCaMP/dLight signal channel) ---
# Cobolt 06-MLD 473 nm diode-pumped solid-state laser

laser_473_model = ExcitationSourceModel(
    name="cobolt_473nm_model",
    manufacturer="Cobolt",
    model_number="06-MLD 473nm",
    description="473 nm DPSS laser for GCaMP/dLight excitation (signal channel)",
    source_type="laser",
    excitation_mode="one-photon",
    wavelength_range_in_nm=[473.0, 473.0],
)
nwbfile.add_device_model(laser_473_model)

laser_473 = ExcitationSource(
    name="laser_473nm",
    description="473 nm laser for dLight1.3b excitation (signal channel)",
    serial_number="COB-2024-00901",
    model=laser_473_model,
    power_in_W=0.05e-3,   # 50 µW at fiber tip (typical for photometry)
    intensity_in_W_per_m2=3.98e2,  # ~400 W/m2 at 400 µm fiber tip
)
nwbfile.add_device(laser_473)

# --- Excitation source 2: 405 nm laser (isosbestic reference channel) ---
# Cobolt 06-MLD 405 nm — excites motion/hemodynamic artifacts, not calcium-dependent

laser_405_model = ExcitationSourceModel(
    name="cobolt_405nm_model",
    manufacturer="Cobolt",
    model_number="06-MLD 405nm",
    description=(
        "405 nm DPSS laser for isosbestic excitation (motion artifact control channel). "
        "GCaMP fluorescence at this wavelength is calcium-independent."
    ),
    source_type="laser",
    excitation_mode="one-photon",
    wavelength_range_in_nm=[405.0, 405.0],
)
nwbfile.add_device_model(laser_405_model)

laser_405 = ExcitationSource(
    name="laser_405nm",
    description="405 nm laser for isosbestic reference channel",
    serial_number="COB-2024-01001",
    model=laser_405_model,
    power_in_W=0.02e-3,   # 20 µW at fiber tip
    intensity_in_W_per_m2=1.59e2,
)
nwbfile.add_device(laser_405)

# --- Optical fiber model: Doric MFC_400/430-0.48 ---
# 400 µm core, 0.48 NA multimode fiber for deep brain photometry

fiber_model_fp = OpticalFiberModel(
    name="doric_mfc_400_model",
    manufacturer="Doric Lenses",
    model_number="MFC_400/430-0.48_4mm_MF2.5_FLT",
    description=(
        "400 µm core diameter, 0.48 NA multimode fiber patch cord "
        "with metal ferrule (2.5 mm diameter, 4 mm active length)"
    ),
    numerical_aperture=0.48,
    core_diameter_in_um=400.0,
    active_length_in_mm=4.0,
    ferrule_name="Metal Ferrule",
    ferrule_model="MF2.5",
    ferrule_diameter_in_mm=2.5,
)
nwbfile.add_device_model(fiber_model_fp)

fiber_insertion_fp = StereotacticPosition(
    name="fiber_insertion",
    anatomical_target="Nucleus accumbens (NAc), core",
    origin="bregma",
    orientation="RAS",
    x_in_mm=1.3,
    y_in_mm=1.3,
    z_in_mm=-4.3,    # 0.2 mm above injection site
    pitch_in_deg=0.0,
    yaw_in_deg=0.0,
    roll_in_deg=0.0,
)

optical_fiber_fp = OpticalFiber(
    name="optical_fiber",
    description="Chronically implanted fiber for NAc dopamine photometry",
    serial_number="FIB-2024-01101",
    model=fiber_model_fp,
    fiber_insertion=fiber_insertion_fp,
)
nwbfile.add_device(optical_fiber_fp)

# --- Dichroic mirror: Semrock FF425-Di01 ---
# 425 nm longpass, separates 405 nm excitation from GCaMP emission

dichroic_model_fp = DichroicMirrorModel(
    name="ff425_di01_model",
    manufacturer="Semrock",
    model_number="FF425-Di01",
    description="425 nm longpass dichroic for separating 405/473 nm excitation from emission",
    cut_on_wavelength_in_nm=425.0,
    reflection_band_in_nm=[350.0, 414.0],
    transmission_band_in_nm=[436.0, 800.0],
    angle_of_incidence_in_degrees=45.0,
)
nwbfile.add_device_model(dichroic_model_fp)

dichroic_fp = DichroicMirror(
    name="dichroic_mirror",
    description="Dichroic mirror separating laser excitation from GCaMP/dLight emission",
    serial_number="DM-2024-01201",
    model=dichroic_model_fp,
)
nwbfile.add_device(dichroic_fp)

# --- Emission filter: Semrock FF03-525/50-25 ---
# 525/50 bandpass for GCaMP/dLight emission

emission_filter_model_fp = BandOpticalFilterModel(
    name="ff03_525_50_fp_model",
    manufacturer="Semrock",
    model_number="FF03-525/50-25",
    description="525/50 nm bandpass emission filter for GCaMP/dLight detection",
    filter_type="Bandpass",
    center_wavelength_in_nm=525.0,
    bandwidth_in_nm=50.0,
)
nwbfile.add_device_model(emission_filter_model_fp)

emission_filter_fp = BandOpticalFilter(
    name="emission_filter",
    description="Bandpass emission filter in photometry detection path",
    serial_number="EMF-2024-01301",
    model=emission_filter_model_fp,
)
nwbfile.add_device(emission_filter_fp)

# --- Photodetector: Newport 2151 femtowatt photoreceiver ---
# Silicon photodetector, transimpedance amplifier, 300–1100 nm

photoreceiver_model_fp = PhotodetectorModel(
    name="newport_2151_model",
    manufacturer="Newport",
    model_number="2151",
    description=(
        "Femtowatt photoreceiver with silicon photodetector, "
        "spectral range 300-1100 nm, variable gain transimpedance amplifier"
    ),
    detector_type="silicon photodiode",
    wavelength_range_in_nm=[300.0, 1100.0],
    gain=1e10,
    gain_unit="V/W",
)
nwbfile.add_device_model(photoreceiver_model_fp)

photoreceiver_fp = Photodetector(
    name="photoreceiver",
    description="Silicon photoreceiver for fiber photometry signal detection",
    serial_number="PHR-2024-01401",
    model=photoreceiver_model_fp,
)
nwbfile.add_device(photoreceiver_fp)

photoreceiver abc.Photodetector at 0x1859854208000
Fields:
  description: Silicon photoreceiver for fiber photometry signal detection
  model: newport_2151_model abc.PhotodetectorModel at 0x1859853333952
Fields:
  description: Femtowatt photoreceiver with silicon photodetector, spectral range 300-1100 nm, variable gain transimpedance amplifier
  detector_type: silicon photodiode
  gain: 10000000000.0
  gain_unit: V/W
  manufacturer: Newport
  model_number: 2151
  wavelength_range_in_nm: [ 300. 1100.]

  serial_number: PHR-2024-01401

In [12]:
with NWBHDF5IO("example_003.nwb", "w") as io:
    io.write(nwbfile)

In [13]:
with NWBHDF5IO("example_003.nwb", "r") as io:
    read_nwbfile = io.read()
    for name, device in read_nwbfile.devices.items():
        print(name, type(device).__name__)

dichroic_mirror DichroicMirror
emission_filter BandOpticalFilter
laser_405nm ExcitationSource
laser_473nm ExcitationSource
optical_fiber OpticalFiber
photoreceiver Photodetector


## 4. Head-Mounted Miniscope (UCLA Miniscope v4)

Head-mounted miniaturized microscopes (miniscopes) allow one-photon fluorescence imaging in freely behaving animals. They use a GRIN (gradient-index) lens to access deep brain structures and a small CMOS sensor for image capture.

**Typical configuration:**
- Integrated 470 nm LED excitation
- GRIN lens relay for deep brain access (e.g., hippocampus, striatum)
- CMOS sensor for imaging
- Emission bandpass filter (integrated in the miniscope body)

In [14]:
nwbfile = NWBFile(
    session_description="Head-mounted miniaturized microscopes (miniscopes) ",
    identifier="example_004",
    session_start_time=datetime.datetime.now(datetime.timezone.utc),
)

In [15]:
# --- Viral vector & indicator ---

viral_vector_ms = ViralVector(
    name="viral_vector_ms",
    description="AAV for pan-neuronal GCaMP7f expression",
    construct_name="AAV9-Syn-GCaMP7f-WPRE",
    manufacturer="Addgene",
    titer_in_vg_per_ml=1.0e13,
)

injection_coords_ms = StereotacticPosition(
    name="viral_injection_coordinates",
    anatomical_target="Dorsal CA1, hippocampus",
    origin="bregma",
    orientation="RAS",
    x_in_mm=1.8,    # lateral
    y_in_mm=-2.3,   # posterior
    z_in_mm=-1.5,   # below surface
)

injection_ms = ViralVectorInjection(
    name="viral_vector_injection",
    description="Hippocampal injection for miniscope GCaMP7f imaging",
    volume_in_uL=0.4,
    injection_date="2024-09-01T11:00:00+00:00",
    viral_injection_coordinates=injection_coords_ms,
    viral_vector=viral_vector_ms,
)

indicator_ms = Indicator(
    name="indicator",
    description="Genetically encoded calcium indicator for miniscope imaging",
    label="GCaMP7f",
    manufacturer="Addgene",
    viral_vector_injection=injection_ms,
)

# --- Excitation source: integrated 470 nm LED (UCLA Miniscope v4) ---
# The Miniscope v4 uses a single surface-mount 470 nm LED (Cree XLamp XP-E2)
# with on-board current control via the Miniscope DAQ

led_model_ms = ExcitationSourceModel(
    name="miniscope_led_model",
    manufacturer="Cree",
    model_number="XLamp XP-E2 (470 nm)",
    description=(
        "470 nm LED integrated in UCLA Miniscope v4 body, "
        "current-controlled via Miniscope DAQ software"
    ),
    source_type="LED",
    excitation_mode="one-photon",
    wavelength_range_in_nm=[455.0, 490.0],
)
nwbfile.add_device_model(led_model_ms)

led_ms = ExcitationSource(
    name="miniscope_led",
    description="Integrated 470 nm LED for GCaMP7f excitation in UCLA Miniscope v4",
    serial_number="MS-LED-2024-01501",
    model=led_model_ms,
    power_in_W=0.1e-3,   # ~0.1 mW, typical operating power
    intensity_in_W_per_m2=1.0e3,
    exposure_time_in_s=33.3e-3,   # ~30 fps, 33 ms exposure
)
nwbfile.add_device(led_ms)

# --- Excitation filter: integrated 469/35 bandpass (Miniscope v4) ---

excitation_filter_model_ms = BandOpticalFilterModel(
    name="miniscope_excitation_filter_model",
    manufacturer="Chroma",
    model_number="ET469/35x",
    description="469/35 nm bandpass excitation filter integrated in Miniscope v4",
    filter_type="Bandpass",
    center_wavelength_in_nm=469.0,
    bandwidth_in_nm=35.0,
)
nwbfile.add_device_model(excitation_filter_model_ms)

excitation_filter_ms = BandOpticalFilter(
    name="excitation_filter",
    description="Excitation bandpass filter for LED cleanup in Miniscope v4",
    serial_number="EF-MS-2024-01601",
    model=excitation_filter_model_ms,
)
nwbfile.add_device(excitation_filter_ms)

# --- Dichroic mirror: 495 nm longpass (integrated in Miniscope v4) ---

dichroic_model_ms = DichroicMirrorModel(
    name="miniscope_dichroic_model",
    manufacturer="Chroma",
    model_number="T495lpxr",
    description="495 nm longpass dichroic mirror integrated in Miniscope v4 optical path",
    cut_on_wavelength_in_nm=495.0,
    reflection_band_in_nm=[400.0, 484.0],
    transmission_band_in_nm=[506.0, 700.0],
    angle_of_incidence_in_degrees=45.0,
)
nwbfile.add_device_model(dichroic_model_ms)

dichroic_ms = DichroicMirror(
    name="dichroic_mirror",
    description="Dichroic mirror in Miniscope v4 fluorescence cube",
    serial_number="DM-MS-2024-01701",
    model=dichroic_model_ms,
)
nwbfile.add_device(dichroic_ms)

# --- Emission filter: 525/39 bandpass (integrated in Miniscope v4) ---

emission_filter_model_ms = BandOpticalFilterModel(
    name="miniscope_emission_filter_model",
    manufacturer="Chroma",
    model_number="ET525/39m",
    description="525/39 nm bandpass emission filter integrated in Miniscope v4",
    filter_type="Bandpass",
    center_wavelength_in_nm=525.0,
    bandwidth_in_nm=39.0,
)
nwbfile.add_device_model(emission_filter_model_ms)

emission_filter_ms = BandOpticalFilter(
    name="emission_filter",
    description="Bandpass emission filter for GCaMP7f detection in Miniscope v4",
    serial_number="EMF-MS-2024-01801",
    model=emission_filter_model_ms,
)
nwbfile.add_device(emission_filter_ms)

# --- GRIN lens (ObjectiveLens): Inscopix ProView 0.6 mm diameter ---
# 1 mm length GRIN relay lens for deep brain access

grin_model_ms = ObjectiveLensModel(
    name="proview_grin_1mm_model",
    manufacturer="Inscopix",
    model_number="ProView Integrated Lens 0.6 mm x 7.3 mm",
    description=(
        "GRIN relay lens for deep brain access, 0.6 mm outer diameter, "
        "7.3 mm length, for imaging CA1 pyramidal layer"
    ),
    numerical_aperture=0.5,
    magnification=1.0,  # relay lens, not magnifying
)
nwbfile.add_device_model(grin_model_ms)

grin_position_ms = StereotacticPosition(
    name="lens_positioning",
    anatomical_target="Dorsal CA1, hippocampus (stratum pyramidale)",
    origin="bregma",
    orientation="RAS",
    x_in_mm=1.8,
    y_in_mm=-2.3,
    z_in_mm=-1.2,    # GRIN lens tip above stratum pyramidale
    pitch_in_deg=0.0,
    yaw_in_deg=0.0,
    roll_in_deg=0.0,
)

grin_lens_ms = ObjectiveLens(
    name="grin_lens",
    description="Chronically implanted GRIN lens for hippocampal CA1 imaging",
    serial_number="GRIN-2024-01901",
    model=grin_model_ms,
    lens_positioning=grin_position_ms,
)
nwbfile.add_device(grin_lens_ms)

# --- Photodetector: CMOS sensor in UCLA Miniscope v4 ---
# Sony IMX225 or similar 1/3" CMOS sensor (exact sensor varies by Miniscope version)

cmos_model_ms = PhotodetectorModel(
    name="miniscope_cmos_model",
    manufacturer="UCLA Miniscope",
    model_number="Miniscope v4 CMOS",
    description=(
        "Integrated CMOS sensor in UCLA Miniscope v4, "
        "~608x608 pixel imaging area, ~30 fps frame rate"
    ),
    detector_type="CMOS",
    wavelength_range_in_nm=[400.0, 700.0],
)
nwbfile.add_device_model(cmos_model_ms)

cmos_ms = Photodetector(
    name="cmos_sensor",
    description="Integrated CMOS image sensor for calcium imaging in Miniscope v4",
    serial_number="MS-CMOS-2024-02001",
    model=cmos_model_ms,
)
nwbfile.add_device(cmos_ms)


cmos_sensor abc.Photodetector at 0x1859854146128
Fields:
  description: Integrated CMOS image sensor for calcium imaging in Miniscope v4
  model: miniscope_cmos_model abc.PhotodetectorModel at 0x1859854211536
Fields:
  description: Integrated CMOS sensor in UCLA Miniscope v4, ~608x608 pixel imaging area, ~30 fps frame rate
  detector_type: CMOS
  manufacturer: UCLA Miniscope
  model_number: Miniscope v4 CMOS
  wavelength_range_in_nm: [400. 700.]

  serial_number: MS-CMOS-2024-02001

In [16]:
with NWBHDF5IO("example_004.nwb", "w") as io:
    io.write(nwbfile)

In [17]:
with NWBHDF5IO("example_004.nwb", "r") as io:
    read_nwbfile = io.read()
    for name, device in read_nwbfile.devices.items():
        print(name, type(device).__name__)

cmos_sensor Photodetector
dichroic_mirror DichroicMirror
emission_filter BandOpticalFilter
excitation_filter BandOpticalFilter
grin_lens ObjectiveLens
miniscope_led ExcitationSource
